In [1]:
import argparse
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


import os
import numpy as np
import torch
from argparse import ArgumentParser
import tqdm 

from config import *
from helpers import * 
import articulate as art
from constants import MODULES
from utils.model_utils import load_model
from data import PoseDataset
from models import MobilePoserNet

import argparse
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset



## dataset

In [ ]:

dataset_name = "dip"

if dataset_name not in datasets.finetune_datasets:
    raise ValueError(f"Test dataset: {dataset_name} not found.")
dataset = PoseDataset( evaluate=dataset_name)

print(f"Starting evaluation: {dataset_name.capitalize()}")

device = "cuda"

# load data
xs, ys = zip(*[(imu.to(device), (pose.to(device), tran)) for imu, pose, joint, tran in dataset])

offline_errs, online_errs = [], []
tran_errors = {window_size: [] for window_size in list(range(1, 8))}


with torch.no_grad():
    for x, y in tqdm(list(zip(xs, ys))):
        pose_t, tran_t = y

x0 = torch.cat([pose_t.to(device), tran_t.to(device)], dim=-1)  # (B, 147)


Loading datasets for TRAIN mode (finetune=None, evaluate=dip)
Datasets: ['dip_test.pt', 'dip_train.pt']



  0%|          | 0/2 [00:00<?, ?it/s]c:\deepika\2_mobileposer\MobilePoser\mobileposer\data.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  file_data = torch.load(data_f

Starting evaluation: Dip


  0%|          | 0/720 [00:00<?, ?it/s]

In [3]:
print(x0.shape)

torch.Size([2783, 147])


In [4]:
def get_dataset(name, n=8000):
    if name == "pose":
        return TensorDataset(x0)
    else:
        raise ValueError(f"Unknown dataset: {name}")

## model

In [5]:


class Block(nn.Module):
    def __init__(self, size: int):
        super().__init__()
        self.ln = nn.LayerNorm(size)
        self.ff = nn.Linear(size, size)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor):
        return x + self.act(self.ff(self.ln(x)))

class SinusoidalEmbedding(nn.Module):
    def __init__(self, size: int, scale: float = 1.0):
        super().__init__()
        self.size = size
        self.scale = scale
        half_size = self.size // 2
        emb = torch.log(torch.Tensor([10000.0])) / (half_size - 1)
        emb = torch.exp(-emb * torch.arange(half_size))
        self.emb = nn.Parameter(emb, requires_grad=False)

    def forward(self, x: torch.Tensor):
        x = x * self.scale
        emb = x * self.emb.unsqueeze(0)
        emb = torch.cat((torch.sin(emb), torch.cos(emb)), dim=-1)
        return emb


class MLP(nn.Module):
    def __init__(self, data_dim=2, hidden_size=128, hidden_layers=3, emb_size=128):
        super().__init__()
        self.data_dim = data_dim

        self.time_emb = SinusoidalEmbedding(emb_size)

        self.concat_size = data_dim + emb_size
        self.data_size = data_dim

        layers = [nn.Linear(self.concat_size, hidden_size), nn.GELU()]
        for _ in range(hidden_layers):
            layers.append(Block(hidden_size))
        layers.append(nn.LayerNorm(hidden_size))
        layers.append(nn.Linear(hidden_size, self.data_size))

        self.joint_mlp = nn.Sequential(*layers)

    def forward(self, x, t):
        t_emb = self.time_emb(t.reshape(-1,1))
        x = torch.cat((x, t_emb), dim=-1)
        return self.joint_mlp(x)

class Diffusion():
    def __init__(self,
                 num_timesteps=1000,
                 beta_start=0.0001,
                 beta_end=0.02,
                 device="cpu"):  

        self.device = device
        self.num_timesteps = num_timesteps

        self.betas = torch.linspace(
            beta_start, beta_end, num_timesteps, dtype=torch.float32, device=device)

        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0)
        self.alphas_cumprod_prev = F.pad(
            self.alphas_cumprod[:-1], (1, 0), value=1.)

        self.sqrt_alphas_cumprod = self.alphas_cumprod ** 0.5
        self.sqrt_one_minus_alphas_cumprod = (1 - self.alphas_cumprod) ** 0.5

        self.sqrt_inv_alphas = torch.sqrt(1. / self.alphas)
        self.noise_coef = self.betas / self.sqrt_one_minus_alphas_cumprod
        self.variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)

    def add_noise(self, x_start, x_noise, timesteps):
        # compute x_t for DDPM algorithm 1
        s1 = self.sqrt_alphas_cumprod[timesteps].reshape(-1, 1)
        s2 = self.sqrt_one_minus_alphas_cumprod[timesteps].reshape(-1, 1)
        return s1 * x_start + s2 * x_noise
    
    def sample_step(self, model_output, timestep, sample):
        # compute x_{t-1} for DDPM algorithm 2
        s1 = self.sqrt_inv_alphas[timestep].reshape(-1, 1)
        s2 = self.noise_coef[timestep].reshape(-1, 1)
        s3 = self.variance[timestep].reshape(-1, 1) ** 0.5
        noise = torch.randn_like(model_output)
        return s1 * (sample - s2 * model_output) + s3 * noise

    def __len__(self):
        return self.num_timesteps



In [6]:
experiment_name = "base"
dataset = "pose"
train_batch_size = 32
eval_batch_size= 1000
num_epochs= 100
learning_rate= 1e-3
num_timesteps= 200
embedding_size= 32
hidden_size= 32
hidden_layers= 1
eval_path= None
device= "cuda"

outdir = f"exps/{experiment_name}"

In [7]:

print(device)
dataset = get_dataset(dataset)
dataloader = DataLoader(
    dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)

model = MLP(
    data_dim=x0.shape[1],
    hidden_size=hidden_size,
    hidden_layers=hidden_layers,
    emb_size=embedding_size).to(device)

diffusion = Diffusion(
    num_timesteps=num_timesteps,
    device=device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: 1 - step / (num_epochs * len(dataloader)))

cuda


In [8]:
if eval_path is None:
    global_step = 0
    losses = []
    print("Training model...")
    for epoch in range(num_epochs):
        model.train()
        progress_bar = tqdm(total=len(dataloader))
        progress_bar.set_description(f"Epoch {epoch}")
        for step, batch in enumerate(dataloader):
            if type(batch) == list:
                batch = batch[0]
            batch = batch.to(device)

            noise = torch.randn_like(batch).to(device)  
            timesteps = torch.randint(
                0, diffusion.num_timesteps, (batch.shape[0],),
                device=device  
            ).long()

            noisy = diffusion.add_noise(batch, noise, timesteps)
            noise_pred = model(noisy.to(device), timesteps.to(device))
            loss = F.mse_loss(noise_pred, noise.to(device))
            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

            progress_bar.update(1)
            logs = {"loss": loss.detach().item(), "step": global_step, "lr": scheduler.get_last_lr()[0]}
            losses.append(loss.detach().item())
            progress_bar.set_postfix(**logs)
            global_step += 1
        progress_bar.close()

    print("Saving model...")
    os.makedirs(outdir, exist_ok=True)
    torch.save(model.state_dict(), f"{outdir}/model.pth")

Training model...


  0%|          | 0/86 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

...

  0%|          | 0/86 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

Saving model...
